[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maercaestro/megat/blob/main/notebooks/06_finetune.ipynb)

# Notebook 06 — Megat GPT-2 355M Finetuning

Finetune the pretrained Megat model on two sources:
- **`maercaestro/cereka-kecil`** — synthetic Malay fiction (500+ stories, LLM-judged)
- **`abuworks`** — your personal fiction (uploaded from Google Drive)

**Base model:** `maercaestro/megat-gpt2-355m`  
**Tokenizer:** `maercaestro/megat-pretrain-data` (custom 32k BPE)  
**Output:** finetuned checkpoint saved to HuggingFace Hub

---
**This notebook:**
1. Installs dependencies and loads base model
2. Loads + tokenizes `cereka-kecil` and your `abuworks` text
3. Mixes both datasets (configurable ratio)
4. Runs full finetuning with per-epoch checkpoints + sample generation
5. Selects the best checkpoint and uploads to HF Hub

**Runtime:** T4 (16GB) or A100 (40GB) — both work. T4 ≈ 45–90 min/epoch.

## Step 1 — Install dependencies

In [ ]:
!pip install -q tokenizers huggingface_hub datasets

## Step 2 — Clone repo and authenticate

In [ ]:
import os
import sys
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

# Clone repo for model.py
if not os.path.exists('megat'):
    !git clone https://github.com/maercaestro/megat.git
else:
    !git -C megat pull --quiet
    print('Repo up to date.')

sys.path.insert(0, 'megat')

from huggingface_hub import login
login(token=HF_TOKEN)
print('Authenticated.')

## Step 3 — Configuration

Adjust these before running. The defaults are sensible starting points for a T4.

In [ ]:
import torch

# ── Device ──────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
print(f'Device: {DEVICE}  ({GPU_NAME},  {VRAM_GB:.1f} GB VRAM)')

# ── Precision ────────────────────────────────────────────────────────
# T4 supports fp16 only. A100/L4 support bf16 (more numerically stable).
USE_BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
DTYPE    = torch.bfloat16 if USE_BF16 else torch.float16
print(f'Precision: {"bf16" if USE_BF16 else "fp16"}')

# ── Batch size ───────────────────────────────────────────────────────
# T4 (16GB):  MICRO_BATCH=2, GRAD_ACCUM=32  → effective batch = 64 sequences
# A100 (40GB): MICRO_BATCH=4, GRAD_ACCUM=16 → effective batch = 64 sequences
MICRO_BATCH  = 2 if VRAM_GB < 30 else 4
GRAD_ACCUM   = 32 if VRAM_GB < 30 else 16
SEQ_LEN      = 1024
print(f'Micro-batch: {MICRO_BATCH}  |  Grad accum: {GRAD_ACCUM}  |  Effective batch: {MICRO_BATCH * GRAD_ACCUM} seqs')

# ── Finetuning hyperparameters ────────────────────────────────────────
PEAK_LR       = 2e-5      # 10–30× lower than pretraining (3e-4). Start here, sweep if needed.
MIN_LR        = 2e-6      # 10% of peak — cosine decays to this floor
WEIGHT_DECAY  = 0.1
GRAD_CLIP     = 1.0
WARMUP_STEPS  = 100       # short warmup — the model is already trained
NUM_EPOCHS    = 5         # 5 epochs is a good starting point; watch for overfitting
DROPOUT       = 0.1       # re-enable during finetuning (unlike pretraining)

# ── Data mix ─────────────────────────────────────────────────────────
# 0.0 = abuworks only  |  1.0 = cereka-kecil only  |  0.2 = 80% abuworks + 20% cereka
SYNTHETIC_RATIO = 0.2     # 20% synthetic data acts as a regulariser — prevents forgetting general Malay

# ── Paths / repos ────────────────────────────────────────────────────
PRETRAIN_CKPT_REPO = 'maercaestro/megat-gpt2-355m'
TOKENIZER_REPO     = 'maercaestro/megat-pretrain-data'
SYNTHETIC_REPO     = 'maercaestro/cereka-kecil'
OUTPUT_REPO        = 'maercaestro/megat-gpt2-355m-finetune'
ABUWORKS_PATH      = 'abuworks.txt'   # uploaded in Step 4
CHECKPOINT_DIR     = 'ft_checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'\nConfig ready.')

## Step 4 — Upload your `abuworks` text

Upload your personal fiction as a single `.txt` file.
Each story should be separated by `<|story|>` / `<|end|>` tags, but plain text works too — the tokenizer will wrap it.

**Option A:** Upload from your computer using the file picker below.  
**Option B:** Mount Google Drive and load directly.

In [ ]:
# ── Option A: Upload from computer ──────────────────────────────────
from google.colab import files
print('Choose your abuworks .txt file:')
uploaded = files.upload()

# Rename to standard path
for fname in uploaded:
    with open(ABUWORKS_PATH, 'wb') as f:
        f.write(uploaded[fname])
    print(f'Saved as {ABUWORKS_PATH}  ({len(uploaded[fname]):,} bytes)')
    break

In [ ]:
# ── Option B: Mount Google Drive (run instead of Option A) ───────────
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/abuworks.txt', ABUWORKS_PATH)
# print('Loaded from Drive.')

## Step 5 — Load the tokenizer

In [ ]:
from huggingface_hub import hf_hub_download
from tokenizers import ByteLevelBPETokenizer

VOCAB_PATH = hf_hub_download(
    repo_id=TOKENIZER_REPO,
    filename='megat_tokenizer/vocab.json',
    repo_type='dataset',
    local_dir='megat_tokenizer',
)
MERGES_PATH = hf_hub_download(
    repo_id=TOKENIZER_REPO,
    filename='megat_tokenizer/merges.txt',
    repo_type='dataset',
    local_dir='megat_tokenizer',
)

tokenizer = ByteLevelBPETokenizer(VOCAB_PATH, MERGES_PATH)

EOT_ID   = tokenizer.token_to_id('<|endoftext|>')
STORY_ID = tokenizer.token_to_id('<|story|>')
END_ID   = tokenizer.token_to_id('<|end|>')

VOCAB_SIZE = tokenizer.get_vocab_size()
print(f'Vocab size:         {VOCAB_SIZE:,}')
print(f'<|endoftext|> ID:   {EOT_ID}')
print(f'<|story|> ID:       {STORY_ID}')
print(f'<|end|> ID:         {END_ID}')

## Step 6 — Load and prepare finetuning data

### 6a. Load `cereka-kecil` synthetic stories

In [ ]:
from datasets import load_dataset

ds = load_dataset(SYNTHETIC_REPO, split='train')
synthetic_texts = [row['text'] for row in ds]   # already formatted as <|story|>...<|end|>

print(f'Synthetic stories loaded: {len(synthetic_texts):,}')
print(f'\nSample (first 200 chars):')
print(synthetic_texts[0][:200])

### 6b. Load and segment `abuworks`

In [ ]:
with open(ABUWORKS_PATH, 'r', encoding='utf-8') as f:
    raw_abuworks = f.read()

print(f'abuworks raw size: {len(raw_abuworks):,} chars')

# Split on existing <|story|> tags if present, otherwise treat the whole file as one document
if '<|story|>' in raw_abuworks:
    # Already formatted — split on <|endoftext|> or double newlines between stories
    parts = [p.strip() for p in raw_abuworks.split('<|endoftext|>') if p.strip()]
    # Ensure each part has story tags
    abuworks_texts = []
    for p in parts:
        if not p.startswith('<|story|>'):
            p = f'<|story|>\n{p}'
        if not p.endswith('<|end|>'):
            p = f'{p}\n<|end|>'
        abuworks_texts.append(p)
else:
    # Plain text — split on two or more blank lines (likely story breaks)
    import re
    parts = [p.strip() for p in re.split(r'\n{3,}', raw_abuworks) if len(p.strip()) > 100]
    abuworks_texts = [f'<|story|>\n{p}\n<|end|>' for p in parts]

print(f'abuworks segments:  {len(abuworks_texts):,}')
print(f'\nSample (first 200 chars):')
print(abuworks_texts[0][:200])

### 6c. Tokenize all documents

In [ ]:
import numpy as np

def tokenize_documents(docs, desc=''):
    """Tokenize a list of strings into a single flat uint16 array."""
    all_ids = []
    for i, doc in enumerate(docs):
        ids = tokenizer.encode(doc).ids
        ids.append(EOT_ID)   # separator between documents
        all_ids.extend(ids)
        if (i + 1) % 100 == 0 or i == len(docs) - 1:
            print(f'  {desc}: {i+1}/{len(docs)}  ({len(all_ids):,} tokens so far)', end='\r')
    print()
    return np.array(all_ids, dtype=np.uint16)

print('Tokenizing synthetic stories...')
synthetic_tokens = tokenize_documents(synthetic_texts, desc='cereka-kecil')
print(f'  → {len(synthetic_tokens):,} tokens  ({len(synthetic_tokens)/1e6:.2f}M)')

print('Tokenizing abuworks...')
abuworks_tokens = tokenize_documents(abuworks_texts, desc='abuworks')
print(f'  → {len(abuworks_tokens):,} tokens  ({len(abuworks_tokens)/1e6:.2f}M)')

### 6d. Build mixed train/val split

In [ ]:
# Mix: sample N synthetic tokens to match SYNTHETIC_RATIO
# e.g. if abuworks = 200k tokens and SYNTHETIC_RATIO=0.2:
#   synthetic slice = 200k * 0.2 / 0.8 = 50k tokens
#   total = 250k tokens, 80% abu / 20% synthetic

n_abu = len(abuworks_tokens)
n_syn_target = int(n_abu * SYNTHETIC_RATIO / max(1 - SYNTHETIC_RATIO, 1e-6))
n_syn_use    = min(n_syn_target, len(synthetic_tokens))

# Shuffle synthetic tokens before slicing
rng = np.random.default_rng(42)
syn_slice = synthetic_tokens[:n_syn_use]

all_tokens = np.concatenate([abuworks_tokens, syn_slice])
rng.shuffle(all_tokens.reshape(-1, 1024)[:len(all_tokens)//1024].view(np.uint16))  # shuffle at chunk level

# 95/5 train/val split
val_size   = max(SEQ_LEN * 4, len(all_tokens) // 20)  # 5% but at least 4 sequences
val_tokens  = all_tokens[:val_size]
train_tokens = all_tokens[val_size:]

print(f'abuworks tokens:    {n_abu:>10,}')
print(f'synthetic tokens:   {n_syn_use:>10,}  ({SYNTHETIC_RATIO*100:.0f}% of mix)')
print(f'Total tokens:       {len(all_tokens):>10,}')
print(f'Train:              {len(train_tokens):>10,}  ({len(train_tokens)//SEQ_LEN:,} full sequences)')
print(f'Val:                {len(val_tokens):>10,}  ({len(val_tokens)//SEQ_LEN:,} full sequences)')

# Save as binary files
train_tokens.tofile('ft_train.bin')
val_tokens.tofile('ft_val.bin')
print('\nSaved ft_train.bin and ft_val.bin')

## Step 7 — Load the pretrained base model

In [ ]:
from model import GPT, GPTConfig
import torch.nn as nn

# Download pretrained checkpoint
CKPT_PATH = hf_hub_download(
    repo_id=PRETRAIN_CKPT_REPO,
    filename='ckpt_final.pt',
    local_dir='megat_base',
)
print(f'Checkpoint: {CKPT_PATH}')

# Build model
config = GPTConfig(
    block_size=SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    n_layer=24,
    n_head=16,
    n_embd=1024,
)
model = GPT(config)

# Load weights
checkpoint = torch.load(CKPT_PATH, map_location='cpu')
if isinstance(checkpoint, dict) and 'model' in checkpoint:
    state_dict = checkpoint['model']
    print(f'Loaded from step {checkpoint.get("step", "?")}, val_loss={checkpoint.get("val_loss", "?")}')
elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

state_dict = {k.replace('_orig_mod.', '').replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

# Enable dropout for finetuning (was 0.0 during pretraining)
# The base GPT doesn't have dropout layers — we add them inline via training mode
# Instead, we manually patch the MLP and Attention modules
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.p = DROPOUT

model = model.to(DEVICE)
model.train()

total_params = sum(p.numel() for p in model.parameters())
print(f'\nModel loaded: {total_params/1e6:.1f}M parameters')
print(f'Device: {DEVICE}')

## Step 8 — Set up optimizer and LR schedule

In [ ]:
import math

optimizer = model.configure_optimizers(
    weight_decay=WEIGHT_DECAY,
    learning_rate=PEAK_LR,
    device=DEVICE,
)

# Compute total steps
steps_per_epoch = max(1, (len(train_tokens) // SEQ_LEN) // (MICRO_BATCH * GRAD_ACCUM))
total_steps     = steps_per_epoch * NUM_EPOCHS
print(f'Steps per epoch: {steps_per_epoch}')
print(f'Total steps:     {total_steps}')

def get_lr(step):
    """Cosine decay with linear warmup."""
    if step < WARMUP_STEPS:
        return PEAK_LR * step / max(1, WARMUP_STEPS)
    if step >= total_steps:
        return MIN_LR
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + (PEAK_LR - MIN_LR) * cosine

print(f'\nLR at step 0:   {get_lr(0):.2e}')
print(f'LR at peak:     {get_lr(WARMUP_STEPS):.2e}')
print(f'LR at end:      {get_lr(total_steps):.2e}')

## Step 9 — Data loader

In [ ]:
class FinetuneDataLoader:
    """
    Simple sequential loader over a flat uint16 token array.
    Returns (x, y) batches of shape (MICRO_BATCH, SEQ_LEN).
    Resets to the beginning each epoch.
    """
    def __init__(self, token_array, batch_size, seq_len):
        self.tokens = token_array
        self.B = batch_size
        self.T = seq_len
        self.pos = 0

    def reset(self):
        self.pos = 0

    def next_batch(self):
        B, T = self.B, self.T
        needed = B * T + 1
        # Wrap around if not enough tokens remain
        if self.pos + needed > len(self.tokens):
            self.pos = 0
        chunk = self.tokens[self.pos : self.pos + needed].astype(np.int64)
        x = torch.from_numpy(chunk[:B*T].reshape(B, T)).to(DEVICE)
        y = torch.from_numpy(chunk[1:B*T+1].reshape(B, T)).to(DEVICE)
        self.pos += B * T
        return x, y

    @property
    def num_batches(self):
        return len(self.tokens) // (self.B * self.T)


train_loader = FinetuneDataLoader(train_tokens, MICRO_BATCH, SEQ_LEN)
val_loader   = FinetuneDataLoader(val_tokens,   MICRO_BATCH, SEQ_LEN)

print(f'Train batches per epoch: {train_loader.num_batches:,}')
print(f'Val   batches:           {val_loader.num_batches:,}')

## Step 10 — Generation helper (for mid-training samples)

In [ ]:
@torch.no_grad()
def sample(prompt, max_new_tokens=150, temperature=0.8, top_p=0.95):
    model.eval()
    ids   = tokenizer.encode(prompt).ids
    idx   = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    out   = model.generate(idx, max_new_tokens=max_new_tokens,
                           temperature=temperature, top_p=top_p)
    model.train()
    gen = out[0, len(ids):].tolist()
    if EOT_ID in gen:
        gen = gen[:gen.index(EOT_ID)]
    return prompt + tokenizer.decode(gen)


SAMPLE_PROMPTS = [
    '<|story|>\n',
    'Malam itu, dia tidak sepatutnya masuk ke dalam bilik itu.',
    '"Kau ingat aku tak tahu apa yang kau buat?" suara itu datang dari belakang.',
]

print('Sample helper ready. Testing on base model...')
print('─' * 60)
print(sample(SAMPLE_PROMPTS[0]))
print('─' * 60)

## Step 11 — Training loop

Trains for `NUM_EPOCHS` epochs. After each epoch:
- Computes validation loss
- Saves a checkpoint
- Generates sample text so you can evaluate quality by eye

**Watch for overfitting:** if val loss starts rising while train loss keeps falling, stop at that epoch's checkpoint.

In [ ]:
import time
from contextlib import nullcontext

scaler      = torch.cuda.amp.GradScaler(enabled=not USE_BF16)  # only needed for fp16
autocast_ctx = torch.autocast(device_type='cuda', dtype=DTYPE) if DEVICE == 'cuda' else nullcontext()

global_step  = 0
best_val_loss = float('inf')
epoch_history = []   # (epoch, train_loss, val_loss)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loader.reset()
    epoch_start   = time.time()
    epoch_loss    = 0.0
    epoch_batches = 0

    model.train()
    optimizer.zero_grad()

    for step in range(steps_per_epoch):
        t0 = time.time()

        # ── Gradient accumulation ──────────────────────────────────
        micro_loss = 0.0
        for micro in range(GRAD_ACCUM):
            x, y = train_loader.next_batch()
            with autocast_ctx:
                _, loss = model(x, y)
                loss    = loss / GRAD_ACCUM
            if USE_BF16:
                loss.backward()
            else:
                scaler.scale(loss).backward()
            micro_loss += loss.item()

        # ── Optimizer step ─────────────────────────────────────────
        lr = get_lr(global_step)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        if USE_BF16:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        else:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

        optimizer.zero_grad()

        epoch_loss    += micro_loss
        epoch_batches += 1
        global_step   += 1

        dt = time.time() - t0
        tps = (MICRO_BATCH * GRAD_ACCUM * SEQ_LEN) / dt

        if step % 20 == 0 or step == steps_per_epoch - 1:
            print(f'  Epoch {epoch}/{NUM_EPOCHS}  step {step+1}/{steps_per_epoch}'
                  f'  loss={micro_loss:.4f}  lr={lr:.2e}  {tps:,.0f} tok/s', end='\r')

    avg_train_loss = epoch_loss / max(epoch_batches, 1)
    print()  # newline after \r

    # ── Validation ─────────────────────────────────────────────────
    model.eval()
    val_loader.reset()
    val_loss_total = 0.0
    val_steps      = min(val_loader.num_batches, 20)  # cap at 20 val batches
    with torch.no_grad():
        for _ in range(val_steps):
            xv, yv = val_loader.next_batch()
            with autocast_ctx:
                _, vl = model(xv, yv)
            val_loss_total += vl.item()
    avg_val_loss = val_loss_total / max(val_steps, 1)

    # ── Checkpoint ─────────────────────────────────────────────────
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch:02d}.pt')
    torch.save({
        'epoch':      epoch,
        'step':       global_step,
        'model':      model.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'train_loss': avg_train_loss,
        'val_loss':   avg_val_loss,
        'config':     config.__dict__,
    }, ckpt_path)

    is_best = avg_val_loss < best_val_loss
    if is_best:
        best_val_loss = avg_val_loss
        import shutil
        shutil.copy(ckpt_path, os.path.join(CHECKPOINT_DIR, 'best.pt'))

    elapsed = time.time() - epoch_start
    epoch_history.append((epoch, avg_train_loss, avg_val_loss))

    print(f'  Epoch {epoch}/{NUM_EPOCHS}  '
          f'train_loss={avg_train_loss:.4f}  '
          f'val_loss={avg_val_loss:.4f}  '
          f'time={elapsed/60:.1f}min  '
          f'{"[BEST]" if is_best else ""}')

    # ── Sample generation ───────────────────────────────────────────
    print('  Sample:')
    print('  ' + '─' * 56)
    text = sample(SAMPLE_PROMPTS[epoch % len(SAMPLE_PROMPTS)])
    for line in text[:400].split('\n'):
        print(f'  {line}')
    print('  ' + '─' * 56)
    print()

print('Training complete.')
print(f'Best val loss: {best_val_loss:.4f} → ft_checkpoints/best.pt')

## Step 12 — Training summary

In [ ]:
print('Epoch  Train Loss  Val Loss')
print('─' * 32)
for epoch, tl, vl in epoch_history:
    marker = ' ← best' if abs(vl - best_val_loss) < 1e-6 else ''
    print(f'  {epoch:2d}     {tl:.4f}      {vl:.4f}{marker}')

# Overfitting diagnostic
if len(epoch_history) >= 2:
    last_tl, last_vl = epoch_history[-1][1], epoch_history[-1][2]
    prev_vl          = epoch_history[-2][2]
    if last_vl > prev_vl + 0.05:
        print('\n⚠  Val loss increased last epoch — you may be overfitting.')
        print('   Consider using the checkpoint from a previous epoch.')
    elif last_tl < 0.5:
        print('\n⚠  Train loss is very low (<0.5) — check if model is memorising training data.')
        print('   Generate samples and compare against your original stories.')
    else:
        print('\n✓  Loss curves look healthy.')

## Step 13 — Load best checkpoint and evaluate

Run this to load the best checkpoint (by val loss) and generate a wider set of samples for manual evaluation.

In [ ]:
best_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best.pt'), map_location=DEVICE)
model.load_state_dict(best_ckpt['model'])
model.eval()

print(f'Loaded best checkpoint:')
print(f'  Epoch:      {best_ckpt["epoch"]}')
print(f'  Train loss: {best_ckpt["train_loss"]:.4f}')
print(f'  Val loss:   {best_ckpt["val_loss"]:.4f}')

In [ ]:
# Generate 5 story openings for manual review
eval_prompts = [
    '<|story|>\n',
    'Malam itu, dia tidak sepatutnya masuk ke dalam bilik itu.',
    '"Kau ingat aku tak tahu apa yang kau buat?" suara itu datang dari belakang.',
    'Kilang itu patut tutup pada pukul 10 malam. Tapi lampu di tingkat 3 masih menyala.',
    'Bukan pertama kali dia dengar bunyi itu. Tapi malam ni berbeza.',
]

print('═' * 60)
print('  Finetuned model samples (best checkpoint)')
print('═' * 60)
for i, prompt in enumerate(eval_prompts, 1):
    text = sample(prompt, max_new_tokens=200, temperature=0.82, top_p=0.95)
    print(f'\n[{i}] Prompt: {prompt[:60]!r}')
    print('─' * 60)
    print(text[:500])
    print()

## Step 14 — Temperature sweep on best checkpoint

Find the sweet spot between coherence and creativity for your model.

In [ ]:
sweep_prompt = 'Dia buka pintu bilik itu dengan perlahan.'

print(f'Prompt: {sweep_prompt!r}\n')
for temp in [0.6, 0.75, 0.85, 1.0]:
    text = sample(sweep_prompt, max_new_tokens=120, temperature=temp, top_p=0.95)
    print(f'── temperature={temp} ──────────────────────────────────')
    print(text[:300])
    print()

## Step 15 — Upload finetuned model to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Create output repo if it doesn't exist
try:
    api.create_repo(OUTPUT_REPO, repo_type='model', private=False)
    print(f'Created: {OUTPUT_REPO}')
except Exception:
    print(f'Repo exists: {OUTPUT_REPO}')

# Upload best checkpoint
api.upload_file(
    path_or_fileobj=os.path.join(CHECKPOINT_DIR, 'best.pt'),
    path_in_repo='ckpt_finetune_best.pt',
    repo_id=OUTPUT_REPO,
    token=HF_TOKEN,
)
print('Uploaded: ckpt_finetune_best.pt')

# Upload all epoch checkpoints
for fname in sorted(os.listdir(CHECKPOINT_DIR)):
    if fname.endswith('.pt') and fname != 'best.pt':
        api.upload_file(
            path_or_fileobj=os.path.join(CHECKPOINT_DIR, fname),
            path_in_repo=fname,
            repo_id=OUTPUT_REPO,
            token=HF_TOKEN,
        )
        print(f'Uploaded: {fname}')

print(f'\nAll checkpoints at: https://huggingface.co/{OUTPUT_REPO}')

## Step 16 — Interactive generation

Try your own prompts.

In [ ]:
# ── Customise these ───────────────────────────────────────────────
MY_PROMPT      = '<|story|>\nTajuk: Malam Terakhir Di Asrama\nGenre: Seram\n\n'
MAX_NEW_TOKENS = 400
TEMPERATURE    = 0.82
TOP_P          = 0.95
# ─────────────────────────────────────────────────────────────────

print('─' * 60)
print(sample(MY_PROMPT, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, top_p=TOP_P))
print('─' * 60)

## Step 17 — Optional: re-run with different hyperparameters

After your first run, go back to **Step 3** and try these variants:

| Round | LR | Dropout | Synthetic ratio | Notes |
|-------|----|---------|-----------------|-------|
| 1 (default) | 2e-5 | 0.1 | 0.2 | Baseline |
| 2 | 1e-5 | 0.1 | 0.2 | Slower, more conservative |
| 3 | 5e-5 | 0.1 | 0.2 | Faster style transfer, riskier |
| 4 | 2e-5 | 0.1 | 0.0 | abuworks only — maximum style capture |
| 5 | 2e-5 | 0.1 | 0.5 | Heavy synthetic mixing — more general |

Pick the best checkpoint across all runs by comparing val loss and sample quality.